In [ ]:
# repository setup runs in the next cell

In [ ]:
import os
import sys
from pathlib import Path

if Path('/kaggle/input').exists():
    print('Kaggle environment detected. Cloning repository...')
    os.system('git clone https://github.com/dvydinh/smpPrediction.git /kaggle/working/repo')
    sys.path.append('/kaggle/working/repo')
    os.chdir('/kaggle/working/repo')
    print('Repository cloned.')

os.system(f'{sys.executable} -m pip install -q -r requirements.txt')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import get_data_paths, load_and_preprocess_data
from src.feature_engineering import add_engineered_features
from src.feature_policy import select_production_features
from src import model_utils
from src import evaluation

DATA_ROOT, OUTPUT_DIR = get_data_paths()
print(f'Data root: {DATA_ROOT}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
df = load_and_preprocess_data(DATA_ROOT)

In [ ]:
df = add_engineered_features(df)

In [ ]:
feature_cols = select_production_features(df)
print(f'Feature count: {len(feature_cols)}')


In [ ]:
model, selected_features = model_utils.train_and_save_model(
    df, feature_cols,
    output_dir=str(OUTPUT_DIR / 'models'),
    model_name='stacking_ensemble.pkl'
)


In [ ]:
evaluation.evaluate_and_plot(
    model, df, selected_features,
    output_dir=str(OUTPUT_DIR / 'eval')
)


In [ ]:
artifact_paths = {
    'model': OUTPUT_DIR / 'models' / 'stacking_ensemble.pkl',
    'metrics': OUTPUT_DIR / 'eval' / 'metrics.txt',
    'manifest': OUTPUT_DIR / 'eval' / 'model_manifest.json',
}
missing = [str(path) for path in artifact_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing Kaggle artifacts: {missing}')

print('Artifacts remain in Kaggle output and will not be pushed to git.')
for name, path in artifact_paths.items():
    print(f'{name}: {path}')